In [1]:
# single branch GRU training script
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
import argparse
import sys
import os
import matplotlib.pyplot as plt

src_path = os.path.abspath(os.path.join(os.getcwd(), 'many_to_many', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)
    
from utils import create_sliding_windows_m2m, SequentialDeepONetDataset, train_val_test_split
from s_deeponet import SequentialDeepONet

torch.manual_seed(0)
np.random.seed(0)

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

   Static hostname: gpua061.delta.ncsa.illinois.edu
         Icon name: computer-server
           Chassis: server
        Machine ID: 4216c320c33b02efc2510a15f446eb32
           Boot ID: 1336515662fd49beb5e803cc76cd9caf
  Operating System: ]8;;https://www.redhat.com/Red Hat Enterprise Linux 8.8 (Ootpa)]8;;
       CPE OS Name: cpe:/o:redhat:enterprise_linux:8::baseos
            Kernel: Linux 4.18.0-477.95.1.el8_8.x86_64
      Architecture: x86-64
Using device: cuda


In [3]:
data_path = '/path/Cosmic10000/data'

target_file = 'dose/combined.npy'
branch_file = 'neutron/neutron_data_2001_2023.npy'
trunc_file = 'coord/grid_array.npy'
target = np.load(os.path.join(data_path,target_file))
input_data = np.load(os.path.join(data_path,branch_file))
trunk = np.load(os.path.join(data_path,trunc_file))

In [4]:
# Normalize trunk input
trunk[:, 0] = (trunk[:, 0] - np.min(trunk[:, 0])) / (np.max(trunk[:, 0]) - np.min(trunk[:, 0]))
trunk[:, 1] = (trunk[:, 1] - np.min(trunk[:, 1])) / (np.max(trunk[:, 1]) - np.min(trunk[:, 1]))

# Assuming input_data and target are defined elsewhere in the notebook
train_input, train_target, val_input, val_target, test_input, test_target = train_val_test_split(input_data, target)

Train input shape: (4017, 12)
Validation input shape: (4018, 12)
Test input shape: (365, 12)


In [5]:
# input data normalization (min-max scaling)
scaler = MinMaxScaler()

train_input = scaler.fit_transform(train_input)
val_input = scaler.transform(val_input)
# test_input = scaler.transform(test_input)

In [6]:
# target data normalization (min-max scaling)
scaler_target = MinMaxScaler()

train_target = scaler_target.fit_transform(train_target)[..., np.newaxis]
val_target = scaler_target.transform(val_target)[..., np.newaxis]
# test_target = scaler_target.transform(test_target)[..., np.newaxis]

In [7]:
window_size = 14
pred_window = 14

# Generate sequences for the training set
train_input_seq, train_target_seq = create_sliding_windows_m2m(train_input, train_target, window_size, pred_window = pred_window)

# # Generate sequences for the testing set
# test_input_seq, test_target_seq = create_sliding_windows_m2m(test_input, test_target, window_size, pred_window = pred_window)

# generate sequences for the validation set
val_input_seq, val_target_seq = create_sliding_windows_m2m(val_input, val_target, window_size, pred_window = pred_window)

# print the shapes of the generated sequences
print("Check the shapes of the generated sequences\n-----------------------------------------")
print("Train input shape:", train_input_seq.shape)
print("Train target shape:", train_target_seq.shape)
print("Validation input shape:", val_input_seq.shape)
print("Validation target shape:", val_target_seq.shape)
# print("Test input shape:", test_input_seq.shape)
# print("Test target shape:", test_target_seq.shape)
print("-----------------------------------------")

Check the shapes of the generated sequences
-----------------------------------------
Train input shape: torch.Size([3990, 14, 12])
Train target shape: torch.Size([3990, 14, 65341, 1])
Validation input shape: torch.Size([3991, 14, 12])
Validation target shape: torch.Size([3991, 14, 65341, 1])
-----------------------------------------


In [8]:
# Create DataLoaders for training and validation sets
print("Create DataLoaders for training and validation sets\n-----------------------------------------")
batch_size = 16
print("Batch size:", batch_size)

train_dataset = SequentialDeepONetDataset(train_input_seq, trunk, train_target_seq)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

val_dataset = SequentialDeepONetDataset(val_input_seq, trunk, val_target_seq)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# test_dataset = SequentialDeepONetDataset(test_input_seq, trunk, test_target_seq)
# test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Create DataLoaders for training and validation sets
-----------------------------------------
Batch size: 16


In [9]:
p, num_outs = train_target_seq.shape[-2], train_target_seq.shape[-1]

# %%
def init_model(model_type = 'gru'):
    dim = 128
    model = SequentialDeepONet(
        branch_type=model_type,
        branch_input_size=12,
        branch_hidden_size=128,
        branch_num_layers=4,
        branch_output_size=dim,
        trunk_architecture=[2, 128, 128, dim],
        num_outputs=num_outs,
        activation_fn=nn.ReLU,
        pred_window = pred_window
    )
    return model

In [10]:
def train_model(model, model_type, train_loader, val_loader=None, device='cpu',
                num_epochs=500, learning_rate=1e-3, patience=10,
                save_path='/saved_models',
                loss_save_path='/loss_history',
                verbose=True):
    """
    Train a PyTorch model with early stopping, validation, and loss logging.

    Parameters:
    - model: PyTorch model
    - train_loader: DataLoader for training
    - val_loader: DataLoader for validation (optional)
    - device: torch.device
    - num_epochs: maximum number of epochs
    - learning_rate: learning rate for optimizer
    - patience: early stopping patience
    - save_path: directory to save the best model
    - loss_save_path: directory to save loss history
    - verbose: whether to print progress
    """
    os.makedirs(save_path, exist_ok=True)
    os.makedirs(loss_save_path, exist_ok=True)

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1E-4)
    criterion = torch.nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min',           # Explicitly set mode
        factor=0.5,           # Reduce LR by half (default is 0.1)
        patience=5,          # Wait 10 epochs before reducing (default is 10)
        threshold=1e-4,       # Minimum change to qualify as improvement
        min_lr=1e-7,          # Minimum learning rate
    )

    best_val_loss = float('inf')
    early_stopping_counter = 0

    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            x_branch, x_trunk, y = [t.to(device) for t in batch]

            optimizer.zero_grad()
            output = model(x_branch, x_trunk)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * y.size(0)

        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        if verbose:
            print(f"[Epoch {epoch+1}/{num_epochs}] Train Loss: {train_loss:.6f}", end='')

        if val_loader is not None:
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    x_branch, x_trunk, y = [t.to(device) for t in batch]
                    output = model(x_branch, x_trunk)
                    loss = criterion(output, y)
                    val_loss += loss.item() * y.size(0)

            val_loss /= len(val_loader.dataset)
            val_losses.append(val_loss)

            scheduler.step(val_loss)

            if verbose:
                print(f" | Val Loss: {val_loss:.6f} | LR: {optimizer.param_groups[0]['lr']:.6e}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                early_stopping_counter = 0
                torch.save(model.state_dict(), os.path.join(save_path, f"{model_type}_best_model_10km_lrschd_{window_size}_{pred_window}.pth"))

            else:
                early_stopping_counter += 1
                if early_stopping_counter >= patience:
                    print("🛑 Early stopping triggered.")
                    break
        else:
            val_losses.append(None)
            print()

    # Save loss history
    np.save(os.path.join(loss_save_path, f"{model_type}_train_loss_10km_lrschd_{window_size}_{pred_window}.npy"), np.array(train_losses))
    if val_loader is not None:
        np.save(os.path.join(loss_save_path, f"{model_type}_val_loss_10km_lrschd_{window_size}_{pred_window}.npy"), np.array(val_losses))

    return model

### Training all models

In [ ]:
#
model_types = ['transformer'] # ['lstm', 'gru', 'fcn', 'transformer']
for model_type in model_types:
    print(f"Training model type: {model_type}")
    torch.manual_seed(0)
    np.random.seed(0)
    # Initialize the model
    model = init_model(model_type).to(device)
    print(model)
    # Train the model
    trained_model = train_model(model=model, model_type=model_type, train_loader=train_loader, val_loader=val_loader, device=device, num_epochs=500, learning_rate=1e-3, patience=10)
    
    print(f"Finished training {model_type} model.\n")

Training model type: transformer
SequentialDeepONet(
  (branch_net): Transformer(
    (input_fc): Linear(in_features=12, out_features=128, bias=True)
    (positional_encoding): PositionalEncoding()
    (transformer_encoder_layer): TransformerEncoderLayer(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
      )
      (linear1): Linear(in_features=128, out_features=2048, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (linear2): Linear(in_features=2048, out_features=128, bias=True)
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout1): Dropout(p=0.1, inplace=False)
      (dropout2): Dropout(p=0.1, inplace=False)
    )
    (transformer_encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
        

: 